# Metabolic Health Explorer: Modelos iniciales

**Objetivo:** entrenar y comparar modelos supervisados que ordenen a los adultos según su probabilidad de tener diabetes o prediabetes, usando medidas de bajo costo para priorizar el tamizaje.


**Criterio de éxito (pregunta de negocio):** capturar al menos el 75 % de los casos tamizando como máximo al 40 % de la población, y superar la regla de solo edad (72,3 % en el EDA)

### Importaciones

In [45]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import SplineTransformer, PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint
from sklearn.metrics import roc_auc_score, average_precision_score


RANDOM_STATE = 42

In [4]:
RUTA = Path("..") / "data" / "raw" / "NHANES_consolidado.csv"

df_nhanes = pd.read_csv(RUTA)
adultos = df_nhanes[df_nhanes["Edad"] >= 18].copy()

print("Total de filas", len(df_nhanes))
print("Adultos", len(adultos))
print("Positivos", adultos["Diabetes"].sum())
print("Tasa de positivos", round(adultos["Diabetes"].mean() * 100, 1), "%")

Total de filas 9763
Adultos 6110
Positivos 902
Tasa de positivos 14.8 %


Se cargan los datos consolidados y se filtra a los adultos (≥ 18 años), porque entre los menores solo el 0,5 % es positivo. Se obtienen 6.110 registros con 902 positivos (14,8 %), igual que en el EDA.

In [5]:
n_ceros = (adultos["Presion_Diastolica"] < 1).sum()
print("Diastólicas en ~0:", n_ceros)

adultos.loc[adultos["Presion_Diastolica"] < 1, "Presion_Diastolica"] = np.nan

# Género: 1 = Hombre, 2 = Mujer en NHANES. Lo pasamos a 0/1 para que sea binaria
adultos["Mujer"] = (adultos["Genero"] == 2).astype(int)

# Porcentaje de nulos por columna
print(adultos.isna().mean().mul(100).round(1).sort_values(ascending=False))

Diastólicas en ~0: 29
Presion_Diastolica    11.7
Presion_Sistolica     11.2
Cintura_cm             8.6
Colesterol_Total       7.7
Colesterol_HDL         7.7
IMC                    4.4
Ingreso_Hogar          1.5
Ingreso_Familia        1.4
Diabetes               0.0
Edad                   0.0
Genero                 0.0
Raza_Etnia             0.0
Mujer                  0.0
dtype: float64


**Limpieza:**
- **Presión diastólica ≈ 0:** los ceros se encuentran como `5.4e-79`. Por eso se filtran con `< 1` y no con `== 0`. Se convierten en nulos en lugar de eliminar la fila, para no perder el resto de la información de esa persona.

- **Género:** NHANES lo codifica como 1 = Hombre y 2 = Mujer. Se recodifica en la variable binaria `Mujer` (0/1).

In [6]:
VARS_SIN_LAB = ["Edad", "Mujer", "IMC", "Cintura_cm",
                "Presion_Sistolica", "Presion_Diastolica"]

VARS_CON_LAB = VARS_SIN_LAB + ["Colesterol_Total", "Colesterol_HDL"]

OBJETIVO = "Diabetes"

X = adultos[VARS_CON_LAB] 
y = adultos[OBJETIVO]

print(X.shape, y.shape)

(6110, 8) (6110,)


Se definen dos conjuntos de variables para comparar:

**Sin laboratorio:** Edad, Mujer, IMC, Cintura, Presión sistólica y diastólica. Son medidas que se toman en cualquier consulta, sin muestra de sangre. 

**Con laboratorio:** Las variables "Sin laboratorio" + Colesterol total y HDL. Colesterol y HDL requieren muestra de sangre. 

`Raza_Etnia` e ingresos se excluyen del modelo: se usan para describir a la población, no para decidir a quién priorizar, por consideraciones de equidad.

### Separar entrenamiento y prueba

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,          
    stratify=y,            
    random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape, ", positivos:", y_train.sum(),
      ", tasa:", round(y_train.mean() * 100, 1), "%")
print("Prueba:", X_test.shape,  ", positivos:", y_test.sum(),
      ", tasa:", round(y_test.mean() * 100, 1), "%")

Entrenamiento: (4888, 8) , positivos: 722 , tasa: 14.8 %
Prueba: (1222, 8) , positivos: 180 , tasa: 14.7 %


Se reserva el 20 % de los datos como conjunto de prueba, con partición estratificada (`stratify=y`) para que ambos conjuntos tengan la misma proporción de positivos (~14,8 %).

El conjunto de prueba no se usa para comparar ni ajustar modelos: se usa una sola vez, al final, con el modelo elegido.

### Métrica de sensibilidad para el tamizaje al 40% de la población

Si solo se puede tamizar al 40 % de la población, empezando por quienes el modelo considera de mayor riesgo, ¿qué porcentaje de todos los casos se encontraría?

In [11]:
def recall_at_k(y_true, y_score, k=0.40):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n_tamizar = int(np.ceil(k * len(y_true)))           
    orden = np.argsort(-y_score, kind="stable")         
    top = orden[:n_tamizar]                            

    return y_true[top].sum() / y_true.sum()             

Cálculo:
1. Ordenar a las personas de mayor a menor probabilidad predicha.
2. Tomar el 40 % superior.
3. Dividir los positivos dentro de ese grupo entre el total de positivos.

In [12]:
# Prueba 1: la edad como puntaje
print("Solo edad:", round(recall_at_k(y, X["Edad"]) * 100, 1), "%")

# Prueba 2: puntaje aleatorio
rng = np.random.default_rng(RANDOM_STATE)
print("Aleatorio:", round(recall_at_k(y, rng.random(len(y))) * 100, 1), "%")

# Prueba 3: puntaje perfecto
print("Perfecto:", round(recall_at_k(y, y) * 100, 1), "%")

Solo edad: 72.3 %
Aleatorio: 38.7 %
Perfecto: 100.0 %



Escala de referencia: azar ≈ 40 %, solo edad ≈ 72 %, meta ≥ 75 %, orden perfecto = 100 %.

#### Convertir la métrica en scorer

In [14]:
scorer_r40 = make_scorer(recall_at_k, response_method="predict_proba")

### Modelos de referencia con cross-validation

In [16]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

METRICAS = {
    "recall_40": scorer_r40,
    "roc_auc": "roc_auc", 
    "pr_auc": "average_precision",
}

Se usa validación cruzada estratificada de 5 particiones sobre el conjunto de entrenamiento: se entrena con 4 partes y se evalúa en la quinta, rotando. Así se obtienen 5 resultados por modelo, que se resumen con su media y desviación estándar. 

Métricas:
- recall@40 %: criterio de negocio.
- ROC-AUC: calidad del ordenamiento en toda la lista (azar = 0,5).
- PR-AUC: se enfoca en la clase positiva y es más informativa con desbalance (azar ≈ tasa de positivos, 0,148).

In [17]:
resultados = [] 

def evaluar(nombre, modelo, columnas):
    cv_res = cross_validate(modelo, X_train[columnas], y_train,
                            cv=cv, scoring=METRICAS)
    fila = {"modelo": nombre, "n_vars": len(columnas)}
    for m in METRICAS:
        fila[f"{m}_media"] = cv_res[f"test_{m}"].mean()
        fila[f"{m}_std"] = cv_res[f"test_{m}"].std()
    resultados.append(fila)
    return pd.DataFrame([fila]).round(3)

#### Modelo Dummy

In [19]:
dummy = DummyClassifier(strategy="prior")
evaluar("Dummy", dummy, VARS_SIN_LAB)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.5,0.0,0.148,0.0


#### Solo edad

In [21]:
solo_edad = LogisticRegression()
evaluar("Solo edad", solo_edad, ["Edad"])

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017


Regresión logística con una sola variable. Como la probabilidad crece con la edad, produce exactamente el mismo orden que ordenar por edad.

In [22]:
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017


El Dummy queda en el nivel del azar (recall@40 % = 0,378; ROC-AUC = 0,5). La regla de solo edad logra 0,715 ± 0,029, algo menos que el 72,3 % del EDA porque ahora se mide en particiones de ~980 personas no vistas al entrenar.

## Regresión Logística

Se construye un `Pipeline` con tres pasos: 
1. Imputación por mediana
2. Estandarización
3. Regresión logística.

Decisiones:
- Estandarizar: las variables tienen escalas distintas (edad 18-80, colesterol 100-300). Estandarizarlas hace que los coeficientes sean comparables y que la regularización las trate por igual.

- `class_weight="balanced"`: da más peso a los positivos durante el entrenamiento (~5,8 veces).

In [24]:
def pipeline_logistica(class_weight="balanced", C=1.0):
    return Pipeline([
        ("imputar", SimpleImputer(strategy="median")),   
        ("escalar", StandardScaler()), 
        ("modelo", LogisticRegression(class_weight=class_weight,
                                      C=C, max_iter=1000)),
    ])

In [25]:
evaluar("Logística sin lab", pipeline_logistica(), VARS_SIN_LAB)
evaluar("Logística con lab", pipeline_logistica(), VARS_CON_LAB)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034


La logística sin laboratorio alcanza un recall@40 % de 0,770, por encima de la meta del 75 % y unos 5 puntos sobre la regla de solo edad. Con laboratorio sube a 0,795: el colesterol aporta unos 2,5 puntos adicionales, a costa de requerir una muestra de sangre.

#### Experimento IMC y cintura

In [34]:
sin_cintura = [v for v in VARS_SIN_LAB if v != "Cintura_cm"]
sin_imc     = [v for v in VARS_SIN_LAB if v != "IMC"]

evaluar("Logística sin lab - solo IMC",    pipeline_logistica(), sin_cintura)
evaluar("Logística sin lab - solo cintura", pipeline_logistica(), sin_imc)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034
6,Logística sin lab - sin class_weight,6,0.765,0.046,0.784,0.021,0.360,0.034
7,Splines sin lab,6,0.769,0.022,0.793,0.015,0.367,0.026
8,Splines + interacción sin lab,6,0.770,0.028,0.792,0.016,0.367,0.028
9,Splines + interacción con lab,8,0.798,0.033,0.808,0.015,0.416,0.031


IMC y cintura tienen una correlación de 0,91 en el EDA. Se prueba si conviene conservar solo una de las dos.
Las tres versiones logran un desempeño prácticamente igual (~0,767-0,770), dentro de la variación de la validación cruzada. Las dos variables aportan casi la misma información. La cintura es ligeramente más informativa y tiene un coeficiente mayor.

#### Experimento class_weight

In [27]:
evaluar("Logística sin lab - sin class_weight",
        pipeline_logistica(class_weight=None), VARS_SIN_LAB)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034
6,Logística sin lab - sin class_weight,6,0.765,0.046,0.784,0.021,0.360,0.034


La diferencia es mínima (0,770 frente a 0,765). La ponderación desplaza todas las probabilidades hacia arriba, pero casi no cambia el orden de las personas, que es lo único que mide recall@40 %.

In [28]:
modelo_sin = pipeline_logistica().fit(X_train[VARS_SIN_LAB], y_train)
modelo_con = pipeline_logistica().fit(X_train[VARS_CON_LAB], y_train)

coefs = pd.DataFrame({
    "sin_lab": pd.Series(modelo_sin.named_steps["modelo"].coef_[0], index=VARS_SIN_LAB),
    "con_lab": pd.Series(modelo_con.named_steps["modelo"].coef_[0], index=VARS_CON_LAB),
}).round(3)
coefs.sort_values("con_lab", key=abs, ascending=False)

,sin_lab,con_lab
Edad,0.913,0.951
Cintura_cm,0.595,0.511
Colesterol_HDL,NaN,-0.379
Colesterol_Total,NaN,-0.221
Mujer,0.062,0.179
Presion_Sistolica,0.152,0.173
IMC,0.113,0.084
Presion_Diastolica,-0.066,-0.016


Interpretación:
- Edad (~0,9) y cintura (~0,5-0,6) son las variables más importantes, en coherencia con el EDA.
- El IMC queda con un coeficiente pequeño porque la cintura captura casi toda su información (colinealidad). No significa que el IMC no se relacione con la diabetes.
- La presión diastólica queda cerca de 0 y es candidata a eliminarse.
- HDL negativo: el HDL alto es protector; el signo es el esperado.
- Colesterol total negativo: es contraintuitivo y probablemente refleja el tratamiento. Las personas ya diagnosticadas suelen tomar estatinas, que reducen el colesterol. El modelo aprende "colesterol bajo → probablemente ya diagnosticado". Es una asociación estadística útil, no una relación causal, y refuerza la preferencia por el escenario sin laboratorio.


### Regresión Logística con no linealidades

La logística supone que el riesgo crece de forma lineal con cada variable. En el EDA, la tasa de casos crece de forma no uniforme con la edad (1,6 % en 18-29 años y ~30 % en 60-69).

Se usan splines (`SplineTransformer`), que dividen la variable en tramos y ajustan una curva suave en cada uno. Se aplican a Edad y Cintura, las dos variables más fuertes según los coeficientes. 

El `ColumnTransformer` permite aplicar un tratamiento distinto a cada grupo de columnas: splines a unas y estandarización a las demás.

In [30]:
def pipeline_splines(columnas, vars_spline=("Edad", "Cintura_cm"),
                     interaccion=False, C=1.0, n_knots=5):
    vars_spline = list(vars_spline)
    vars_lineales = [c for c in columnas if c not in vars_spline]

    transformaciones = [
        ("splines", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(n_knots=n_knots, degree=3)),
        ]), vars_spline),

        ("lineales", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), vars_lineales),
    ]

    if interaccion:
        transformaciones.append(
            ("interaccion", Pipeline([
                ("imputar", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler()),
                ("producto", PolynomialFeatures(degree=2, interaction_only=True,
                                                include_bias=False)),
            ]), ["Edad", "Cintura_cm"])
        )

    return Pipeline([
        ("preparar", ColumnTransformer(transformaciones)),
        ("modelo", LogisticRegression(class_weight="balanced", C=C, max_iter=2000)),
    ])

In [31]:
evaluar("Splines sin lab", pipeline_splines(VARS_SIN_LAB), VARS_SIN_LAB)
evaluar("Splines + interacción sin lab",
        pipeline_splines(VARS_SIN_LAB, interaccion=True), VARS_SIN_LAB)
evaluar("Splines + interacción con lab",
        pipeline_splines(VARS_CON_LAB, interaccion=True), VARS_CON_LAB)

pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034
6,Logística sin lab - sin class_weight,6,0.765,0.046,0.784,0.021,0.360,0.034
7,Splines sin lab,6,0.769,0.022,0.793,0.015,0.367,0.026
8,Splines + interacción sin lab,6,0.770,0.028,0.792,0.016,0.367,0.028
9,Splines + interacción con lab,8,0.798,0.033,0.808,0.015,0.416,0.031


Los splines mejoran el ROC-AUC (0,783 - 0,793) y el PR-AUC, pero el recall@40 % se mantiene en ~0,77. 

Esto pasa porque el ROC-AUC evalúa el orden en toda la lista, mientras que el recall@40 % solo mide si cada caso queda por encima o por debajo del corte del 40 %. Los splines reordenan mejor a personas lejos del corte, pero casi no mueven a nadie a través de él.

La interacción Edad × Cintura tampoco aporta mejora: en escala log-odds, el efecto conjunto de edad y cintura ya queda capturado como suma de efectos.

#### Ajuste de hiperparámetros

In [33]:
grid = GridSearchCV(
    pipeline_splines(VARS_SIN_LAB, interaccion=True),
    param_grid={
        "modelo__C": [0.01, 0.1, 1, 10],
        "preparar__splines__spline__n_knots": [3, 4, 5, 7],
    },
    scoring=scorer_r40,
    cv=cv,
)
grid.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor recall@40 %:", round(grid.best_score_, 3))

(pd.DataFrame(grid.cv_results_)
   [["params", "mean_test_score", "std_test_score"]]
   .sort_values("mean_test_score", ascending=False)
   .head(6))

Mejores parámetros: {'modelo__C': 0.01, 'preparar__splines__spline__n_knots': 4}
Mejor recall@40 %: 0.774


,params,mean_test_score,std_test_score
1,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.774301,0.032194
9,"{'modelo__C': 1, 'preparar__splines__spline__n...",0.772902,0.027409
0,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.771561,0.041735
2,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.771523,0.029570
12,"{'modelo__C': 10, 'preparar__splines__spline__...",0.771513,0.026947
8,"{'modelo__C': 1, 'preparar__splines__spline__n...",0.771504,0.024432


Se usa `GridSearchCV` para ajustar la fuerza de regularización (`C`; un valor menor regulariza más) y el número de nodos de los splines (`n_knots`; más nodos dan una curva más flexible). La selección se hace con el scorer de recall@40 %.

## Modelos de árboles

- **Random Forest:** cientos de árboles independientes, cada uno entrenado con una muestra distinta; se promedian sus predicciones.
- **HistGradientBoosting (HGB):** árboles en secuencia, donde cada uno corrige los errores del anterior. 

Ninguno necesita estandarizar, porque los árboles dividen por umbrales y la escala no cambia el orden de los valores.

### Sin ajuste

In [48]:
hgb_default = HistGradientBoostingClassifier(class_weight="balanced",
                                             random_state=RANDOM_STATE)

rf_default = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),   
    ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                      random_state=RANDOM_STATE, n_jobs=-1)),
])

for nombre, modelo in [("HGB por defecto", hgb_default), ("RF por defecto", rf_default)]:
    res = cross_validate(modelo, X_train[VARS_SIN_LAB], y_train, cv=cv,
                         scoring=METRICAS, return_train_score=True)
    print(f"{nombre}: recall@40 % entrenamiento = {res['train_recall_40'].mean():.3f}, "
          f"validación = {res['test_recall_40'].mean():.3f}")


HGB por defecto: recall@40 % entrenamiento = 1.000, validación = 0.733
RF por defecto: recall@40 % entrenamiento = 1.000, validación = 0.738


**Diagnóstico de sobreajuste:** con sus parámetros por defecto, ambos modelos logran un recall@40 % perfecto (1,000) en entrenamiento, pero solo 0,733 (HGB) y 0,738 (RF) en validación, por debajo de la regresión logística (0,770). Los árboles crecen hasta separar casi a cada persona y memorizan ruido en lugar de patrones generalizables. Es necesario restringir su complejidad mediante el ajuste de hiperparámetros.

### Ajustar HistGradientBoosting

In [49]:
busqueda_hgb = RandomizedSearchCV(
    HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    param_distributions={
        "learning_rate":     loguniform(0.01, 0.3),   
        "max_iter":          randint(50, 400),        
        "max_depth":         [2, 3, 4, None],         
        "min_samples_leaf":  randint(20, 200),        
        "l2_regularization": loguniform(1e-3, 10),    
    },
    n_iter=30,              
    scoring=scorer_r40,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
busqueda_hgb.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", busqueda_hgb.best_params_)
print("Mejor recall@40 %:", round(busqueda_hgb.best_score_, 3))

Mejores parámetros: {'l2_regularization': 0.026730883107816707, 'learning_rate': 0.026000059117302653, 'max_depth': None, 'max_iter': 90, 'min_samples_leaf': 176}
Mejor recall@40 %: 0.778


Se usa `RandomizedSearchCV`, que prueba 30 combinaciones sorteadas en lugar de una grilla completa: con 5 hiperparámetros, una grilla de 4 valores cada uno serían 1.024 combinaciones.

El hiperparámetro clave es `min_samples_leaf`: obliga a que cada hoja tenga un mínimo de personas, lo que impide memorizar individuos. El mejor modelo resultó muy restringido (`min_samples_leaf` ≈ 176, `learning_rate` ≈ 0,026), señal de que la relación en los datos es simple.


### Ajustar Random Forest

In [50]:
busqueda_rf = RandomizedSearchCV(
    rf_default,
    param_distributions={
        "modelo__max_depth":        [3, 4, 5, 6, 8, None],
        "modelo__min_samples_leaf": randint(5, 100),
        "modelo__max_features":     ["sqrt", 0.5, None],  
    },
    n_iter=20,
    scoring=scorer_r40,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
busqueda_rf.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", busqueda_rf.best_params_)
print("Mejor recall@40 %:", round(busqueda_rf.best_score_, 3))


Mejores parámetros: {'modelo__max_depth': 5, 'modelo__max_features': 'sqrt', 'modelo__min_samples_leaf': 25}
Mejor recall@40 %: 0.773


Se ajustan la profundidad máxima, el tamaño mínimo de hoja y el número de variables que considera cada división. El mejor modelo usa `max_depth = 5` y `min_samples_leaf = 25`, también mucho más restringido que el modelo por defecto.

### Agregar a tabla de resultados

In [51]:
evaluar("HGB ajustado sin lab", busqueda_hgb.best_estimator_, VARS_SIN_LAB)
evaluar("RF ajustado sin lab",  busqueda_rf.best_estimator_,  VARS_SIN_LAB)

hgb_con_lab = HistGradientBoostingClassifier(class_weight="balanced",
                                             random_state=RANDOM_STATE,
                                             **busqueda_hgb.best_params_)
evaluar("HGB ajustado con lab", hgb_con_lab, VARS_CON_LAB)

pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034
6,Logística sin lab - sin class_weight,6,0.765,0.046,0.784,0.021,0.360,0.034
7,Splines sin lab,6,0.769,0.022,0.793,0.015,0.367,0.026
8,Splines + interacción sin lab,6,0.770,0.028,0.792,0.016,0.367,0.028
9,Splines + interacción con lab,8,0.798,0.033,0.808,0.015,0.416,0.031


Una vez ajustados, los árboles alcanzan 0,778 (HGB) y 0,773 (RF) sin laboratorio, prácticamente iguales a la regresión logística (0,770); la diferencia es menor que la desviación estándar. El 0,778 del HGB es además levemente optimista, porque sus hiperparámetros se eligieron con estas mismas particiones.

Con estas variables no hay interacciones complejas que la logística no capture: la señal se concentra en edad, cintura y presión sistólica, y cualquier modelo razonable la encuentra.

## Comparación y elección de modelo

In [43]:
tabla = (pd.DataFrame(resultados)
           .drop_duplicates(subset="modelo", keep="last")     
           .sort_values("recall_40_media", ascending=False)
           .reset_index(drop=True))

tabla["supera_meta"] = tabla["recall_40_media"] >= 0.75
tabla.round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std,supera_meta
0,Splines + interacción con lab,8,0.798,0.033,0.808,0.015,0.416,0.031,True
1,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034,True
2,HGB ajustado con lab,8,0.780,0.028,0.803,0.015,0.422,0.029,True
3,HGB ajustado sin lab,6,0.778,0.025,0.786,0.017,0.363,0.023,True
4,RF ajustado sin lab,6,0.773,0.019,0.789,0.017,0.371,0.023,True
5,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036,True
6,Splines + interacción sin lab,6,0.770,0.028,0.792,0.016,0.367,0.028,True
7,Splines sin lab,6,0.769,0.022,0.793,0.015,0.367,0.026,True
8,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034,True
9,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035,True


In [44]:
mejor = tabla.loc[0]
print(f"Mejor modelo: {mejor['modelo']} → {mejor['recall_40_media']:.3f} ± {mejor['recall_40_std']:.3f}")

sin_lab = tabla[tabla["n_vars"] <= 6]
mejor_sin = sin_lab.iloc[0]
umbral = mejor_sin["recall_40_media"] - mejor_sin["recall_40_std"]

print(f"\nMejor sin lab: {mejor_sin['modelo']} ({mejor_sin['recall_40_media']:.3f})")
print(f"Modelos sin lab estadísticamente empatados (≥ {umbral:.3f}):")
print(sin_lab[sin_lab["recall_40_media"] >= umbral][["modelo", "recall_40_media", "recall_40_std"]].round(3))

Mejor modelo: Splines + interacción con lab → 0.798 ± 0.033

Mejor sin lab: HGB ajustado sin lab (0.778)
Modelos sin lab estadísticamente empatados (≥ 0.754):
                                  modelo  recall_40_media  recall_40_std
3                   HGB ajustado sin lab            0.778          0.025
4                    RF ajustado sin lab            0.773          0.019
5                      Logística sin lab            0.770          0.042
6          Splines + interacción sin lab            0.770          0.028
7                        Splines sin lab            0.769          0.022
8       Logística sin lab - solo cintura            0.767          0.042
9           Logística sin lab - solo IMC            0.767          0.035
10  Logística sin lab - sin class_weight            0.765          0.046


Las ocho variantes sin laboratorio obtienen un recall@40 % entre 0,765 y 0,778. La diferencia entre la mejor y la peor (0,013) es menor que la desviación estándar de cualquiera de ellas (0,019-0,046), así que con estos datos no es posible distinguirlas. Aplicando la regla de una desviación estándar, se elige el modelo más simple entre los empatados.

**Modelo seleccionado: regresión logística sin laboratorio**
- Interpretabilidad: sus coeficientes permiten explicar el efecto de cada variable al equipo de salud pública.
- Simplicidad de despliegue: un pipeline pequeño, rápido y estable.
- Coherencia con la pregunta de negocio: usa solo medidas no invasivas.
- Comparación justa: no se ajustaron hiperparámetros sobre las mismas particiones, a diferencia del HGB, cuyo 0,778 es levemente optimista.

El escenario con laboratorio supera al sin laboratorio en ~2,5 puntos, pero requiere muestra de sangre y el coeficiente del colesterol total refleja probablemente el efecto del tratamiento, no una relación con el riesgo.

#### Entrenar todo el modelo y evaluarlo

In [46]:
# 1. Entrenar con TODO el conjunto de entrenamiento
modelo_final = pipeline_logistica().fit(X_train[VARS_SIN_LAB], y_train)
referencia_edad = LogisticRegression().fit(X_train[["Edad"]], y_train)

# 2. Probabilidades en prueba
prob_final = modelo_final.predict_proba(X_test[VARS_SIN_LAB])[:, 1]
prob_edad = referencia_edad.predict_proba(X_test[["Edad"]])[:, 1]

# 3. Métricas
eval_prueba = pd.DataFrame({
    "modelo": ["Logística sin lab (final)", "Solo edad (referencia)"],
    "recall_40": [recall_at_k(y_test, prob_final), recall_at_k(y_test, prob_edad)],
    "roc_auc": [roc_auc_score(y_test, prob_final), roc_auc_score(y_test, prob_edad)],
    "pr_auc": [average_precision_score(y_test, prob_final), 
               average_precision_score(y_test, prob_edad)],
}).round(3)
eval_prueba

,modelo,recall_40,roc_auc,pr_auc
0,Logística sin lab (final),0.822,0.804,0.355
1,Solo edad (referencia),0.761,0.766,0.281


El modelo seleccionado se reentrena con todo el conjunto de entrenamiento y se evalúa una única vez en el conjunto de prueba, junto con la regla de solo edad sobre las mismas personas, para que la comparación sea justa.

### Intervalo de confianza con Bootstrap

Con 180 positivos en prueba, el resultado puede variar varios puntos según qué personas hayan quedado en esa partición. El bootstrap simula 2.000 conjuntos de prueba alternativos remuestreando con reemplazo y calcula la métrica en cada uno. El rango del 95 % central es el intervalo de confianza.

In [47]:
rng = np.random.default_rng(RANDOM_STATE)
y_t = y_test.to_numpy()
n = len(y_t)

recall_final, diferencia = [], []
for _ in range(2000):
    idx = rng.integers(0, n, n)                       
    r_final = recall_at_k(y_t[idx], prob_final[idx])
    r_edad = recall_at_k(y_t[idx], prob_edad[idx])
    recall_final.append(r_final)
    diferencia.append(r_final - r_edad)

ic_final = np.percentile(recall_final, [2.5, 97.5])
ic_dif = np.percentile(diferencia, [2.5, 97.5])

print(f"recall@40 % del modelo final: IC 95 % = [{ic_final[0]:.3f}, {ic_final[1]:.3f}]")
print(f"Mejora sobre la edad: IC 95 % = [{ic_dif[0]:.3f}, {ic_dif[1]:.3f}]")
print(f"% de remuestreos en que el modelo supera a la edad: {np.mean(np.array(diferencia) > 0):.1%}")


recall@40 % del modelo final: IC 95 % = [0.758, 0.881]
Mejora sobre la edad: IC 95 % = [0.000, 0.115]
% de remuestreos en que el modelo supera a la edad: 96.7%


**Meta de negocio:** el modelo captura el 82,2 % de los casos tamizando al 40 % de la población (IC 95 % bootstrap: 75,8 %-88,1 %). El límite inferior del intervalo supera el 75 %, así que la meta se cumple, aunque con poco margen.

**Comparación con la referencia:** el modelo supera a la regla de solo edad en 6,1 puntos, y la mejora es positiva en el 96,7 % de los remuestreos (IC 95 %: 0,0-11,5 puntos). Hay evidencia razonable de que combinar variables aporta valor sobre la edad sola.